# 🦅 Tutorial Resmi (v6): Penerjemah AI & Penalaran Morfologi Bahasa Indonesia ↔ Bahasa Gorontalo
**Dikembangkan oleh [Panita Research Gorontalo](https://huggingface.co/panitagorontalo)** | Berbasis Korpus Terpadu **v6** Kamus **Prof. Dr. H. Mansoer Pateda (1987 & 2001)**

Notebook Google Colab ini memandu Anda langkah demi langkah untuk menjalankan ketiga model kecerdasan buatan Bahasa Gorontalo **v6**:
1. **[`panitagorontalo/nllb-200-gorontalo-bi-600M`](https://huggingface.co/panitagorontalo/nllb-200-gorontalo-bi-600M)**: Mesin Penerjemah Neural Seq2Seq dua arah (`ind_Latn` ↔ `gor_Latn`, SacreBLEU `40.59`) — ringan dan cepat (bisa di CPU maupun T4 GPU).
2. **[`panitagorontalo/qwen2.5-7b-gorontalo-lora`](https://huggingface.co/panitagorontalo/qwen2.5-7b-gorontalo-lora)**: LLM 7B Instruksi & Penalaran Morfologi Bahasa Gorontalo (4 Siasat Pateda, 4-bit QLoRA muat di T4 15 GB GPU).
3. **[`panitagorontalo/gemma3-27b-gotoai-gorontalo-lora`](https://huggingface.co/panitagorontalo/gemma3-27b-gotoai-gorontalo-lora)**: LLM 27B (`GoTo-AI v3`) terkuat di kelas LLM (SacreBLEU Clean `34.45` `ID→GOR` / `29.20` `GOR→ID`, membutuhkan GPU L4 24 GB 4-bit atau A100 80 GB `bf16`).
4. **Aplikasi Web Gradio (`share=True`)**: Antarmuka web interaktif dengan pemecahan paragraf/kalimat otomatis.

> **Tips Google Colab**: Untuk Model 1 & 2, pilih **Runtime → Change runtime type → T4 GPU (Gratis)**. Untuk Model 3 (27B), gunakan **L4 (24 GB)** atau **A100 (80 GB)**.

## Langkah 1: Instalasi Pustaka Pendukung
Jalankan sel di bawah ini untuk menginstal pustaka `transformers`, `peft`, `accelerate`, `bitsandbytes`, dan `gradio`.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes sentencepiece protobuf gradio

## Langkah 2: Menjalankan Model 1 (`panitagorontalo/nllb-200-gorontalo-bi-600M` v6)
Model ini sangat ringan (~2,46 GB) dan sudah dilengkapi **pemecahan otomatis per paragraf dan per kalimat** agar input panjang tetap diterjemahkan utuh.

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

NLLB_MODEL_ID = "panitagorontalo/nllb-200-gorontalo-bi-600M"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else (torch.float16 if device == "cuda" else torch.float32)

print(f"Memuat {NLLB_MODEL_ID} pada {device.upper()}...")
nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_ID)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL_ID, torch_dtype=dtype).to(device)
nllb_model.eval()

def terjemahkan_nllb(teks: str, arah: str = "id2gor", num_beams: int = 4) -> str:
    """Menerjemahkan kalimat maupun paragraf panjang secara dua arah (id2gor / gor2id)."""
    src_lang = "ind_Latn" if arah == "id2gor" else "gor_Latn"
    tgt_lang = "gor_Latn" if arah == "id2gor" else "ind_Latn"
    src_id = nllb_tokenizer.convert_tokens_to_ids(src_lang)
    tgt_id = nllb_tokenizer.convert_tokens_to_ids(tgt_lang)
    eos_id = nllb_tokenizer.eos_token_id

    paragraf = [p.strip() for p in re.split(r"\n+", teks.strip()) if p.strip()]
    hasil_paragraf = []
    for p in paragraf:
        kalimat = [s.strip() for s in re.split(r"(?<=[.!?])\s+", p) if s.strip()]
        multi = len(kalimat) > 1
        bagian = []
        for kal in kalimat:
            sub_ids = nllb_tokenizer(kal, add_special_tokens=False, truncation=True, max_length=250)["input_ids"]
            inp_ids = torch.tensor([[src_id] + sub_ids + [eos_id]], dtype=torch.long, device=device)
            with torch.inference_mode():
                gen_ids = nllb_model.generate(
                    input_ids=inp_ids,
                    forced_bos_token_id=tgt_id,
                    max_new_tokens=160,
                    num_beams=num_beams,
                    no_repeat_ngram_size=4,
                    early_stopping=True,
                )
            pred = nllb_tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()
            if pred and not pred.endswith((".", "!", "?")):
                akhir = kal[-1] if kal.endswith((".", "!", "?")) else "."
                if multi or akhir in ("?", "!"):
                    pred += akhir
            bagian.append(pred)
        hasil_paragraf.append(" ".join(bagian))
    return "\n\n".join(hasil_paragraf)

# Uji Paragraf Indonesia -> Gorontalo
paragraf_id = (
    "Kemarin pagi ayah pergi ke pasar untuk membeli ikan dan beras. "
    "Di jalan beliau bertemu dengan paman yang baru pulang dari kebun. "
    "Setelah berbelanja, ayah langsung pulang ke rumah karena ibu sudah menunggu di dapur."
)
print("=== PARAGRAF ID -> GOR (NLLB-200-600M v6) ===")
print(terjemahkan_nllb(paragraf_id, arah="id2gor"))

# Uji Kalimat Gorontalo -> Indonesia
for kal in ["Wa:tiya loona:o ode pa:tali ola:ngo.", "Yilongola yio di:la losikola?"]:
    print(f"\nGOR: {kal}\nID : {terjemahkan_nllb(kal, arah='gor2id')}")

## Langkah 3: Menjalankan Model 2 (`panitagorontalo/qwen2.5-7b-gorontalo-lora` v6)
Model LLM 7B ini mendukung **penerjemahan kalimat/paragraf**, **pembentukan kata turunan berimbuhan (Siasat A)**, dan **adaptasi fonologisasi kata serapan vokalis (Siasat B)** dalam mode 4-bit NF4 (~5,5 GB VRAM, muat di T4 GPU gratis).

In [ ]:
import re
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_LLM = "Qwen/Qwen2.5-7B-Instruct"
LORA_ADAPTER = "panitagorontalo/qwen2.5-7b-gorontalo-lora"

SYSTEM_PROMPT = (
    "Anda adalah pakar bahasa dan penerjemah dwibahasa Bahasa Indonesia <-> Bahasa Gorontalo "
    "yang menguasai tata bahasa, 5 sistem kala, morfofonemik, morfem penunjuk arah (mayi, mao, mola, mota), "
    "dan kaidah leksikografi Prof. Dr. H. Mansoer Pateda."
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Memuat Qwen2.5-7B-Instruct + Adapter LoRA Bahasa Gorontalo v6...")
qwen_tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_LLM,
    quantization_config=bnb_config,
    device_map="auto",
)
qwen_model = PeftModel.from_pretrained(base_model, LORA_ADAPTER)
qwen_model.eval()

def tanya_qwen_gorontalo(instruksi: str, teks_input: str, split_kalimat: bool = True) -> str:
    def infer_satu(seg: str) -> str:
        prompt = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{instruksi}\n\nInput: {seg}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        enc = qwen_tokenizer(prompt, return_tensors="pt").to(qwen_model.device)
        with torch.inference_mode():
            gen_ids = qwen_model.generate(
                **enc,
                max_new_tokens=192,
                do_sample=False,
                no_repeat_ngram_size=4,
                pad_token_id=qwen_tokenizer.pad_token_id,
                eos_token_id=qwen_tokenizer.eos_token_id,
            )
        out_ids = gen_ids[0, enc["input_ids"].shape[1]:]
        return qwen_tokenizer.decode(out_ids, skip_special_tokens=True).strip()

    if not split_kalimat:
        return infer_satu(teks_input)
    paragraf = [p.strip() for p in re.split(r"\n+", teks_input.strip()) if p.strip()]
    hasil = []
    for p in paragraf:
        kalimat = [s.strip() for s in re.split(r"(?<=[.!?])\s+", p) if s.strip()]
        bagian = []
        for kal in kalimat:
            pred = infer_satu(kal)
            if pred and not pred.endswith((".", "!", "?")):
                akhir = kal[-1] if kal.endswith((".", "!", "?")) else "."
                if len(kalimat) > 1 or akhir in ("?", "!"):
                    pred += akhir
            bagian.append(pred)
        hasil.append(" ".join(bagian))
    return "\n\n".join(hasil)

print("1. Terjemahan ID -> GOR:")
print(tanya_qwen_gorontalo(
    "Terjemahkan kalimat bahasa Indonesia berikut ke dalam bahasa Gorontalo yang baik dan benar sesuai kaidah Prof. Mansoer Pateda.",
    "Saya pergi ke pasar kemarin."
))
print("\n2. Perakitan Morfologi (Siasat A):")
print(tanya_qwen_gorontalo(
    "Bentuklah kata turunan bahasa Gorontalo dari akar kata 'tuladu' beserta artinya dalam bahasa Indonesia.",
    "Akar kata: tuladu -> Makna sasaran: menulis",
    split_kalimat=False,
))
print("\n3. Fonologisasi Kata Serapan (Siasat B):")
print(tanya_qwen_gorontalo(
    "Adaptasikan kata bahasa Indonesia berikut ke dalam bahasa Gorontalo sesuai kaidah fonotaktik vokalis Prof. Mansoer Pateda beserta contoh imbuhannya.",
    "adil",
    split_kalimat=False,
))

## Langkah 3b (Opsional untuk GPU L4 24 GB / A100): Model 3 (`panitagorontalo/gemma3-27b-gotoai-gorontalo-lora` v6)
Model 27B ini membutuhkan VRAM minimal **~18–20 GB (4-bit QLoRA pada Colab L4)** atau **~64 GB (`bfloat16` pada A100-80GB)**. Lewati sel ini jika Anda memakai T4 (15 GB VRAM).

In [ ]:
# Jalankan hanya pada GPU >= 24 GB VRAM (mis. Colab L4 / A100)
import re, torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_GEMMA = "GoToCompany/gemma3-27b-cpt-gotoai-v3-instruct"
LORA_GEMMA = "panitagorontalo/gemma3-27b-gotoai-gorontalo-lora"

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
if vram_gb < 22:
    print(f"VRAM GPU saat ini {vram_gb:.1f} GB (< 22 GB). Lewati sel Gemma 3 27B dan gunakan NLLB/Qwen.")
else:
    use_bf16 = vram_gb >= 68
    gemma_tok = AutoTokenizer.from_pretrained(LORA_GEMMA)
    if gemma_tok.pad_token is None:
        gemma_tok.pad_token = gemma_tok.eos_token
    kwargs = {"torch_dtype": torch.bfloat16, "device_map": "auto"}
    if not use_bf16:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
        )
    base_g = AutoModelForCausalLM.from_pretrained(BASE_GEMMA, **kwargs)
    gemma_model = PeftModel.from_pretrained(base_g, LORA_GEMMA)
    gemma_model.eval()
    eos_ids = [i for i in [gemma_tok.eos_token_id, gemma_tok.convert_tokens_to_ids("<end_of_turn>")] if isinstance(i, int) and i >= 0]

    def tanya_gemma3_gorontalo(instruksi: str, teks_input: str) -> str:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"{instruksi}\n\nInput: {teks_input}"}]
        prompt = gemma_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = gemma_tok(prompt, add_special_tokens=False, return_tensors="pt").to(gemma_model.device)
        with torch.inference_mode():
            gen_ids = gemma_model.generate(
                **enc, max_new_tokens=192, do_sample=False, no_repeat_ngram_size=4,
                pad_token_id=gemma_tok.pad_token_id, eos_token_id=eos_ids or gemma_tok.eos_token_id,
            )
        return gemma_tok.decode(gen_ids[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    print(tanya_gemma3_gorontalo("Terjemahkan kalimat bahasa Indonesia berikut ke dalam bahasa Gorontalo yang baik dan benar sesuai kaidah Prof. Mansoer Pateda.", "Tiap pagi halaman rumah harus disapu."))

## Langkah 4: Meluncurkan Aplikasi Web Gradio Interaktif (`share=True`)
Jalankan sel di bawah ini untuk membuka antarmuka web **Gradio** lengkap dengan tautan publik (`https://xxxx.gradio.live`) yang dapat diakses dari peramban HP maupun komputer Anda.

In [ ]:
import gradio as gr

def antarmuka_terjemahan(teks, arah, pilihan_model):
    if not teks.strip():
        return ""
    if pilihan_model == "NLLB-200-600M (Cepat & Ringan)":
        kode_arah = "id2gor" if arah == "Indonesia → Gorontalo" else "gor2id"
        return terjemahkan_nllb(teks, arah=kode_arah)
    else:
        if arah == "Indonesia → Gorontalo":
            inst = "Terjemahkan kalimat bahasa Indonesia berikut ke dalam bahasa Gorontalo yang baik dan benar sesuai kaidah Prof. Mansoer Pateda."
        else:
            inst = "Terjemahkan kalimat bahasa Gorontalo berikut ke dalam bahasa Indonesia yang baik dan benar."
        return tanya_qwen_gorontalo(inst, teks, split_kalimat=True)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🦅 Penerjemah AI Bahasa Indonesia ↔ Bahasa Gorontalo (v6 — Panita Research Gorontalo)")
    with gr.Row():
        arah_input = gr.Radio(["Indonesia → Gorontalo", "Gorontalo → Indonesia"], value="Indonesia → Gorontalo", label="Arah Terjemahan")
        model_input = gr.Radio(["NLLB-200-600M (Cepat & Ringan)", "Qwen2.5-7B-LoRA (LLM & Morfologi)"], value="NLLB-200-600M (Cepat & Ringan)", label="Pilih Model AI")
    teks_in = gr.Textbox(lines=4, value="Saya pergi ke pasar kemarin.", label="Teks Masukan (Kalimat maupun Paragraf)")
    tombol = gr.Button("Terjemahkan 🚀", variant="primary")
    teks_out = gr.Textbox(lines=4, label="Hasil Terjemahan")
    tombol.click(fn=antarmuka_terjemahan, inputs=[teks_in, arah_input, model_input], outputs=teks_out)

demo.launch(share=True)